# Room-Temperature Body-Diode ANN

In [ ]:
# Import the room-temperature data loader and body-diode ANN.
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from DeviceData import load_body_diode_characteristics
from body_diode_hybrid import (
    BodyDiodeNN,
    train_ibd_network,
    predict_ibd,
    save_ibd_pkl,
    load_ibd_pkl,
)


## Load the room-temperature body-diode dataset

In [ ]:
# Read every Vgs curve from the 25 C body-diode folder.
BASE_DIR = Path.cwd()
TEMPLATE_DIR = BASE_DIR / "Template"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = load_body_diode_characteristics(TEMPLATE_DIR)

print(df.groupby(["T_C", "Vgs"]).size())
print("Total body-diode points =", len(df))

assert np.allclose(
    df["T_C"].to_numpy(),
    25.0,
)

Vgs = torch.tensor(
    df["Vgs"].to_numpy(np.float32),
    dtype=torch.float32,
).reshape(-1, 1)

Vds = torch.tensor(
    df["Vds"].to_numpy(np.float32),
    dtype=torch.float32,
).reshape(-1, 1)

Ibd = torch.tensor(
    df["Ibd"].to_numpy(np.float32),
    dtype=torch.float32,
).reshape(-1, 1)

MODEL_FILE = BASE_DIR / "Ibd_NN.pkl"

print("Device =", device)
print(
    "Vds range =",
    float(Vds.min()),
    "to",
    float(Vds.max()),
    "V",
)
print(
    "Ibd range =",
    float(Ibd.min()),
    "to",
    float(Ibd.max()),
    "A",
)


## Train Eq. (29) at room temperature

In [ ]:
# Use a longer training run without regional tail weighting.
EPOCHS = 20000
LEARNING_RATE = 1.0e-4
BATCH_SIZE = 128
VAL_FRACTION = 0.20
SEED = 42

SIGN_WEIGHT = 1.0
ZERO_WEIGHT = 1.0
ZERO_WIDTH = 0.50

model, norm, history = train_ibd_network(
    Vgs,
    Vds,
    Ibd,
    device=device,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION,
    seed=SEED,
    print_every=500,
    sign_weight=SIGN_WEIGHT,
    zero_weight=ZERO_WEIGHT,
    zero_width=ZERO_WIDTH,
)

print(
    "Best validation loss =",
    history.best_val_loss,
)


In [ ]:
# Save the trained room-temperature body-diode ANN.
save_ibd_pkl(
    MODEL_FILE,
    model,
    norm,
    training_metadata={
        "temperature_C": 25.0,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "validation_fraction": VAL_FRACTION,
        "seed": SEED,
        "best_validation_loss": float(
            history.best_val_loss
        ),
        "number_of_samples": int(
            len(df)
        ),
        "sign_weight": SIGN_WEIGHT,
        "zero_weight": ZERO_WEIGHT,
        "zero_width": ZERO_WIDTH,
        "regional_tail_weighting": False,
    },
)

print(
    "Saved =",
    MODEL_FILE,
)


## Static-characteristic accuracy

In [ ]:
# Predict every datasheet point and report global and tail errors.
Ibd_pred = predict_ibd(
    model,
    norm,
    Vgs.reshape(-1),
    Vds.reshape(-1),
    device=device,
).cpu().numpy().reshape(-1)

Ibd_true = (
    Ibd.cpu()
    .numpy()
    .reshape(-1)
)

Vds_np = (
    Vds.cpu()
    .numpy()
    .reshape(-1)
)

error = (
    Ibd_pred
    - Ibd_true
)

rmse = float(
    np.sqrt(
        np.mean(
            error ** 2
        )
    )
)

mae = float(
    np.mean(
        np.abs(
            error
        )
    )
)

tail_mask = (
    Vds_np < -5.5
)

tail_rmse = float(
    np.sqrt(
        np.mean(
            error[tail_mask] ** 2
        )
    )
)

print(
    "RMSE =",
    rmse,
    "A",
)

print(
    "MAE =",
    mae,
    "A",
)

print(
    "Tail RMSE for Vds < -5.5 V =",
    tail_rmse,
    "A",
)


In [ ]:
# Report RMSE for each Vgs curve.
for vgs_value in sorted(
    df["Vgs"].unique()
):
    mask = np.isclose(
        df["Vgs"].to_numpy(),
        vgs_value,
    )

    curve_rmse = float(
        np.sqrt(
            np.mean(
                error[mask] ** 2
            )
        )
    )

    print(
        f"Vgs = {vgs_value:g} V | "
        f"RMSE = {curve_rmse:.6f} A"
    )


In [ ]:
# Plot the datasheet points and ANN body-diode curves.
plt.figure(
    figsize=(7.2, 5.2)
)

for vgs_value in sorted(
    df["Vgs"].unique()
):
    measured = df[
        np.isclose(
            df["Vgs"],
            vgs_value,
        )
    ].sort_values(
        "Vds"
    )

    plt.scatter(
        measured["Vds"],
        measured["Ibd"],
        s=20,
    )

    Vds_curve = np.linspace(
        measured["Vds"].min(),
        0.0,
        300,
    )

    Vgs_curve = np.full_like(
        Vds_curve,
        vgs_value,
    )

    Ibd_curve = predict_ibd(
        model,
        norm,
        Vgs_curve,
        Vds_curve,
        device=device,
    ).cpu().numpy().reshape(-1)

    plt.plot(
        Vds_curve,
        Ibd_curve,
        label=rf"$V_{{GS}}={vgs_value:g}$ V",
    )

plt.xlabel(
    r"$V_{DS}$ (V)"
)

plt.ylabel(
    r"$I_{bd}$ (A)"
)

plt.title(
    r"Body-Diode Characteristics at $T_J=25^\circ$C"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Verify that the saved checkpoint reloads independently.
model_loaded, norm_loaded, checkpoint = load_ibd_pkl(
    MODEL_FILE,
    device=device,
)

print(
    checkpoint["model_name"]
)

print(
    checkpoint["architecture"]
)

print(
    checkpoint["input_names"]
)

print(
    checkpoint["training_metadata"]
)


In [ ]:
# Plot the training and validation composite losses.
plt.figure(
    figsize=(6, 4)
)

plt.semilogy(
    history.train_loss,
    label="Training",
)

plt.semilogy(
    history.val_loss,
    label="Validation",
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Composite normalized loss"
)

plt.grid(
    True
)

plt.legend()
plt.tight_layout()
plt.show()
